# S4 - ML distribuido con Spark MLlib (Regresion)

**Actividad:** integrar tres fuentes reales de sensores ambientales en un DataLake analitico particionado (Bronze -> Silver -> Gold), y sobre esa salida entrenar y comparar modelos de regresion distribuida con Spark MLlib, reportando RMSE, R2 y MAE.

Estructura del notebook: cada fase de CRISP-DM es un bloque (## Fase N), y cada actividad dentro de la fase es un paso numerado (## 3.F.M), igual que en la guia.


## Fase 1 — Business Understanding


## 3.1.1 Crear el notebook y la `SparkSession`


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/03 04:13:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"


## 3.1.2 Definir la variable numerica a predecir


**Objetivo:** estimar `Valor_Objetivo` (hoy, `CE`) a partir de otras variables medidas
en el mismo instante — no un pronostico con historial temporal (eso es contenido de S10, 2.3).


## 3.1.3 Definir la decision de negocio asociada


**Alcance:** comparar un modelo base de regresion lineal, tres configuraciones de
regularizacion y un segundo algoritmo (Random Forest), reportando RMSE, R2 y MAE — sin
busqueda exhaustiva de hiperparametros. El modelo ganador es el que decide si vale la pena
sostener un pipeline de regresion distribuida sobre esta fuente, en vez de no predecir nada.


## Fase 2 — Data Understanding


## 3.2.1 Cargar los datos


In [3]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType

schema_ce = StructType([
    StructField("DateTime", TimestampType(), nullable=False),
    StructField("CE", DoubleType(), nullable=True),
])
df_ce = spark.read.csv(f"{ORIGEN_DATOS}/campo_electrico.csv", header=True, schema=schema_ce)

schema_cm = StructType([
    StructField("DateTime", TimestampType(), nullable=False),
    StructField("CM", DoubleType(), nullable=True),
])
df_cm = spark.read.csv(f"{ORIGEN_DATOS}/campo_magnetico.csv", header=True, schema=schema_cm)

schema_va = StructType([
    StructField("TempOut", DoubleType(), nullable=True),
    StructField("OutHum", DoubleType(), nullable=True),
    StructField("WindSpeed", DoubleType(), nullable=True),
    StructField("Bar", DoubleType(), nullable=True),
    StructField("SolarRad", DoubleType(), nullable=True),
    StructField("UVIndex", DoubleType(), nullable=True),
    StructField("DateTime", TimestampType(), nullable=False),
])
df_va = spark.read.csv(f"{ORIGEN_DATOS}/variables_ambientales.csv", header=True, schema=schema_va)


**Error frecuente**: la fuente original trae la columna de radiacion solar como `SolarRad.`
(con un punto al final). Referenciarla luego con `col("SolarRad.")` falla con `AnalysisException`
— basta con declarar el nombre ya limpio (`SolarRad`, sin punto) en el `StructField` de arriba,
como ya se hizo en el schema.


`variables_ambientales.csv` trae `DateTime` duplicada — hay que resolverlo antes de integrar,
porque un `join` contra una clave duplicada multiplica filas del lado que no lo esta:


In [4]:
from pyspark.sql.functions import col, count as spark_count, when, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

columnas_conteo_nulos = [c for c in df_va.columns if c != "DateTime"]

df_va_con_conteo = df_va.withColumn(
    "CantidadNulos",
    sum(when(col(c).isNull(), 1).otherwise(0) for c in columnas_conteo_nulos),
)

ventana_va = Window.partitionBy("DateTime").orderBy(col("CantidadNulos").asc())

df_va_unico = (
    df_va_con_conteo
    .withColumn("row_num", row_number().over(ventana_va))
    .filter(col("row_num") == 1)
    .drop("row_num", "CantidadNulos")
)


`DateTime` es la clave comun para integrar; el campo electrico queda como tabla principal
(`left join`), porque interesa el periodo que ese sensor cubre:


In [5]:
df_integrado = (
    df_ce
    .join(df_cm, on="DateTime", how="left")
    .join(df_va_unico, on="DateTime", how="left")
)

print(f"Integrado: {df_integrado.count():,} registros x {len(df_integrado.columns)} columnas")


26/09/03 04:13:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Integrado: 93,027 registros x 9 columnas


## 3.2.2 Describir las variables


In [6]:
df_integrado.printSchema()

df_integrado.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in df_integrado.columns
]).show(vertical=True, truncate=False)


root
 |-- DateTime: timestamp (nullable = true)
 |-- CE: double (nullable = true)
 |-- CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)



26/09/03 04:13:47 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

-RECORD 0--------
 DateTime  | 0   
 CE        | 0   
 CM        | 0   
 TempOut   | 0   
 OutHum    | 0   
 WindSpeed | 0   
 Bar       | 0   
 SolarRad  | 0   
 UVIndex   | 0   



## 3.2.3 Analizar estadisticas descriptivas


In [7]:
df_integrado.describe(["CE", "CM", "TempOut", "OutHum", "WindSpeed", "Bar", "SolarRad", "UVIndex"]).show()


26/09/03 04:13:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/09/03 04:13:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|                 CE|                CM|           TempOut|           OutHum|        WindSpeed|               Bar|          SolarRad|           UVIndex|
+-------+-------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|              93027|             93027|             93027|            93027|            93027|             93027|             93027|             93027|
|   mean|-1.1499939802422956|24433.059324712114| 16.67085792296833|83.90338288883872|3.054300364410254| 949.5039182172831|129.04232104657788| 0.859116170574133|
| stddev| 0.9653406399186278|3327.6148516126304|2.7215826624050963|5.187489027636735|3.900013901712125|1.4003907801656077|198.04353437182107|1.4949346407925304|
|    min|              -6.74|     

## 3.2.4 Analizar correlacion con el objetivo


In [8]:
predictores_candidatos = [
    c for c in df_integrado.columns
    if c not in ("DateTime", "CE")
]

for columna in predictores_candidatos:
    correlacion = df_integrado.stat.corr(columna, "CE")
    print(f"{columna:12s} correlacion con CE: {correlacion:.4f}")


26/09/03 04:13:52 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

CM           correlacion con CE: 0.0115


26/09/03 04:13:54 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


TempOut      correlacion con CE: 0.1170


26/09/03 04:13:56 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


...

Bar          correlacion con CE: 0.0132


26/09/03 04:14:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


SolarRad     correlacion con CE: -0.0244


26/09/03 04:14:01 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


UVIndex      correlacion con CE: -0.0092


Advertencias sobre este resultado, antes de sacar conclusiones: todavia no se filtro
`CM = 99999` (3.3.1) — ese codigo de error puede distorsionar su correlacion real con `CE` — y
`df_integrado` todavia puede tener nulos sueltos (3.2.2) que Spark ignora en el calculo, no
reemplaza. Esta es una lectura preliminar; la relacion real se confirma recien con `df_valido`
(3.3.2) y, sobre todo, con los coeficientes o la `featureImportances` del modelo entrenado (3.4).


## Fase 3 — Data Preparation


## 3.3.1 Limpiar los datos


`CM = 99999` es un codigo de error de sensor, no un nulo: Spark lo ve como un `Double` valido,
no como `NULL` — por eso `isNull()` no lo detecta, hace falta filtrarlo por el valor de dominio
conocido.


In [9]:
errores_cm = df_integrado.filter(col("CM") == 99999).count()
print(f"Filas con codigo de error CM=99999: {errores_cm:,}")

df_limpio = df_integrado.filter(col("CM") != 99999)
print(f"Filas despues de eliminar el codigo de error: {df_limpio.count():,}")


26/09/03 04:14:03 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


Filas con codigo de error CM=99999: 180


26/09/03 04:14:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


Filas despues de eliminar el codigo de error: 92,847


## 3.3.2 Tratar nulos, errores y duplicados


Primero, confirmar que ni la deduplicacion ni el `join` dejaron `DateTime` repetida:


In [10]:
total_final = df_limpio.count()
sin_duplicar = df_limpio.dropDuplicates(["DateTime"]).count()

print(f"Total: {total_final:,}, sin duplicar por DateTime: {sin_duplicar:,}")
assert total_final == sin_duplicar, "Hay DateTime duplicada en la tabla integrada final"


26/09/03 04:14:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


Total: 92,847, sin duplicar por DateTime: 92,847


Con eso confirmado, quedan los nulos finales sobre las variables fisicas:


In [11]:
VARIABLES_MODELO = [
    "CE", "CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "SolarRad", "UVIndex",
]

antes = df_limpio.count()
df_valido = df_limpio.na.drop(subset=VARIABLES_MODELO)
despues = df_valido.count()

print(f"Filas antes: {antes:,}, despues de na.drop(subset=VARIABLES_MODELO): {despues:,}")
print(f"Filas eliminadas por nulos en variables criticas: {antes - despues:,}")

df_valido = df_valido.cache()


26/09/03 04:14:09 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
26/09/03 04:14:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv


Filas antes: 92,847, despues de na.drop(subset=VARIABLES_MODELO): 92,847
Filas eliminadas por nulos en variables criticas: 0


Con la tabla ya limpia, se persiste como capa Gold particionada por mes, en vez de
dejarla solo en memoria:


In [12]:
from pyspark.sql.functions import date_format

ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

df_particionable = df_valido.withColumn("AnioMes", date_format(col("DateTime"), "yyyy-MM"))

(
    df_particionable
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("AnioMes")
    .save(f"{ARTIFACTS}/campo_electrico_particionado")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/campo_electrico_particionado")):
    print(carpeta)


26/09/03 04:14:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, SolarRad., UVIndex, DateTime
 Schema: TempOut, OutHum, WindSpeed, Bar, SolarRad, UVIndex, DateTime
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

._SUCCESS.crc
AnioMes=2026-05
AnioMes=2026-06
AnioMes=2026-07
AnioMes=2026-08
AnioMes=2026-09
_SUCCESS


Se lee de vuelta para confirmar que no hubo perdida de filas, y que Spark usa
`PartitionFilters` en el plan de ejecucion al filtrar por `AnioMes`:


In [13]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/campo_electrico_particionado")
df_verificacion.printSchema()

assert df_verificacion.count() == df_particionable.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

df_verificacion.filter(col("AnioMes") == "2026-09").explain(True)


root
 |-- DateTime: timestamp (nullable = true)
 |-- CE: double (nullable = true)
 |-- CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)

Verificado: 92,847 filas, ida y vuelta sin perdida.
== Parsed Logical Plan ==
'Filter '`=`('AnioMes, 2026-09)
+- Relation [DateTime#3093,CE#3094,CM#3095,TempOut#3096,OutHum#3097,WindSpeed#3098,Bar#3099,SolarRad#3100,UVIndex#3101,AnioMes#3102] parquet

== Analyzed Logical Plan ==
DateTime: timestamp, CE: double, CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, SolarRad: double, UVIndex: double, AnioMes: string
Filter (AnioMes#3102 = 2026-09)
+- Relation [DateTime#3093,CE#3094,CM#3095,TempOut#3096,OutHum#3097,WindSpeed#3098,Bar#3099,SolarRad#3100,UVIndex#3101,AnioMes#3102] parque

Para ver cuantas filas quedaron guardadas en cada particion, sin salir de Spark ni
contar archivos a mano:


In [14]:
df_verificacion.groupBy("AnioMes").count().orderBy("AnioMes").show(truncate=False)


+-------+-----+
|AnioMes|count|
+-------+-----+
|2026-05|28107|
|2026-06|41502|
|2026-07|1409 |
|2026-08|20582|
|2026-09|1247 |
+-------+-----+



In [15]:
df_valido.unpersist()


DataFrame[DateTime: timestamp, CE: double, CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, SolarRad: double, UVIndex: double]

`df_verificacion` ya es la salida Gold, leida y verificada — no hace falta volver a leer
el Parquet desde disco para empezar la parte de modelado:


In [16]:
df = df_verificacion

df.printSchema()
print(f"Filas: {df.count():,}")
df.describe(VARIABLES_MODELO).show()


root
 |-- DateTime: timestamp (nullable = true)
 |-- CE: double (nullable = true)
 |-- CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)

Filas: 92,847
+-------+-------------------+-----------------+------------------+-----------------+------------------+------------------+-----------------+------------------+
|summary|                 CE|               CM|           TempOut|           OutHum|         WindSpeed|               Bar|         SolarRad|           UVIndex|
+-------+-------------------+-----------------+------------------+-----------------+------------------+------------------+-----------------+------------------+
|  count|              92847|            92847|             92847|            92847|             92847|             92

## De esta salida a dos preguntas distintas: regresion (S4) y series de tiempo (S10)

`campo_electrico_particionado/` no tiene un solo destino. La misma tabla alimenta dos
sesiones que le hacen a los datos preguntas de naturaleza distinta:

| | S4 — Regresion (hoy) | S10 — Series de tiempo |
|---|---|---|
| Pregunta de fondo | Que otras variables explican `CE`? | El pasado de `CE` predice su futuro? |
| Predictores | Las otras 7 variables, en el mismo instante `t` | `CE` en instantes anteriores (`t`, `t-1`, ...) |
| Objetivo | `CE` en ese mismo instante `t` | `CE` en un instante futuro (`t+1`) |
| Orden de las filas | Irrelevante — cada fila es independiente | Critico — el orden cronologico es el dato |
| Division entrenamiento/prueba | Aleatoria (`randomSplit`) | Cronologica (equivalente a `TimeSeriesSplit`) |
| Riesgo si se usa la division del otro caso | Ninguno | Fuga de informacion: el modelo "veria" el futuro al entrenar |


## 3.3.3 Seleccionar predictores


In [17]:
PREDICTORES = [v for v in VARIABLES_MODELO if v != "CE"]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")


Predictores (7): ['CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'SolarRad', 'UVIndex']


## 3.3.4 Ensamblar el vector de predictores (`VectorAssembler`)


In [18]:
from pyspark.ml.feature import VectorAssembler

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", "CE")

df_ml.show(5, truncate=False)


+---------------------------------------+-----+
|features                               |CE   |
+---------------------------------------+-----+
|[24250.9,15.0,83.0,0.0,948.3,0.0,0.0]  |-1.83|
|[24351.3,18.1,86.0,6.4,947.6,402.0,2.4]|-2.32|
|[24272.9,15.7,85.0,0.0,950.3,0.0,0.0]  |-1.85|
|[24293.4,19.2,84.0,0.0,948.7,0.0,0.0]  |-1.11|
|[24257.2,15.1,84.0,0.0,949.7,0.0,0.0]  |-2.35|
+---------------------------------------+-----+
only showing top 5 rows


## 3.3.5 Dividir aleatoriamente en entrenamiento y prueba


In [19]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


Entrenamiento: 74,434 filas
Prueba: 18,413 filas


## 3.3.6 Escalar si aplica


"Si aplica" es literal aqui, y en Spark MLlib no aplica: `LinearRegression` tiene el
parametro `standardization` (`True` por defecto) — ya estandariza los predictores
internamente antes de ajustar el modelo, precisamente para que `regParam` (3.4.3) penalice
de forma justa entre variables de escalas muy distintas (`CM` en miles, `TempOut`/`OutHum`
en decenas), y devuelve los coeficientes en la escala original de `features`, no en unidades
estandarizadas (documentacion oficial de Spark). Escalar a mano con `StandardScaler` antes
de entrenar seria trabajo redundante. `RandomForestRegressor` (3.4.4) tampoco lo necesita
— sus arboles dividen por umbrales, no por magnitud de coeficientes.

Este paso no siempre "no aplica": en librerias que no estandarizan internamente, escalar a
mano sigue siendo necesario — vale la pena confirmarlo en la documentacion del modelo
concreto, no asumirlo por costumbre. Por eso no hay celda de codigo aqui: no hace falta
ninguna transformacion adicional, `df_train`/`df_test` siguen igual.


## Fase 4 — Modeling


## 3.4.1 Entrenar regresion lineal


In [20]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="CE")
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


26/09/03 04:14:29 WARN Instrumentation: [03c6ed90] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
[Stage 167:=================================>                      (6 + 4) / 10]

Coeficientes: [-0.0026969906496403283,0.1241169169986803,0.010430245366843716,-0.07398119937843363,-0.010412796360618586,2.3180026207879208e-05,0.019429197441476413]
Intercepto: 71.50011725494751


**Advertencias esperadas, no errores**: pueden aparecer `regParam is zero, which might
cause numerical instability and overfitting` (es la linea base a proposito, sin regularizar)
y `netlib-blas: JNI_OnLoad...` (falta una libreria nativa, Spark usa JVM pura — no afecta
resultados).


## 3.4.2 Evaluar con RMSE, R2 y MAE


In [21]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select("CE", "prediction").show(5)

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(labelCol="CE", predictionCol="prediction", metricName=metrica)
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(predicciones_base, "LinearRegression base")


+-----+--------------------+
|   CE|          prediction|
+-----+--------------------+
|-2.38|-0.08317586373686936|
|-2.32| -0.1702886617202637|
| -2.1|-0.20367314181770269|
|-2.07|-0.06746274099312188|
|-1.07|-0.36581976244217174|
+-----+--------------------+
only showing top 5 rows
LinearRegression base: RMSE=0.9153  R2=0.1119  MAE=0.7365


## 3.4.3 Probar regularizacion e hiperparametros basicos


In [22]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="CE",
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


26/09/03 04:14:34 WARN Instrumentation: [4083d836] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularizacion: RMSE=0.9153  R2=0.1119  MAE=0.7365
Ridge (L2): RMSE=0.9175  R2=0.1075  MAE=0.7417
Elastic Net (L1+L2): RMSE=0.9260  R2=0.0909  MAE=0.7563


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.915263,0.111896,0.736459
1,Ridge (L2),0.917524,0.107502,0.741710
2,Elastic Net (L1+L2),0.925996,0.090943,0.756304


La fila "Sin regularizacion" es una verificacion util: deberia salir igual al modelo
base de 3.4.2 — si no coincide, algo cambio entre celdas.


## 3.4.4 Entrenar un modelo basado en arboles


In [23]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="CE", numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(df_train)


26/09/03 04:14:51 WARN DAGScheduler: Broadcasting large task binary with size 1030.9 KiB
26/09/03 04:14:53 WARN DAGScheduler: Broadcasting large task binary with size 1931.8 KiB
                                                                                

## 3.4.5 Evaluar con RMSE, R2 y MAE


In [24]:
predicciones_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(predicciones_rf, "Random Forest")


Random Forest: RMSE=0.7464  R2=0.4094  MAE=0.5747


## 3.4.6 Analizar importancia de variables


`RandomForestRegressor` calcula, sin costo adicional, `featureImportances`: una proporcion
de cuanto reduce cada variable el error del modelo en promedio — las proporciones de las 7
variables suman 1.0.


In [25]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

for variable, importancia in importancias:
    print(f"{variable:12s} {importancia:.4f}")


WindSpeed    0.2549
TempOut      0.2426
OutHum       0.1550
CM           0.1523
Bar          0.0998
SolarRad     0.0528
UVIndex      0.0426


## Fase 5 — Evaluation


## 3.5.1 Comparar los modelos candidatos


In [26]:
comparacion_final = pd.DataFrame(comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
])[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")


,Configuracion,RMSE,R2,MAE
3,Random Forest,0.746406,0.409360,0.574693
0,Sin regularizacion,0.915263,0.111896,0.736459
1,Ridge (L2),0.917524,0.107502,0.741710
2,Elastic Net (L1+L2),0.925996,0.090943,0.756304


## 3.5.2 Seleccionar el mejor modelo


Con base en la tabla de la celda anterior, elige cual configuracion tuvo el mejor RMSE en
tu propia corrida. El nombre de variable `modelo_ganador` de 3.6.1 asume que fue Random
Forest — ajustalo segun tu resultado real.


## 3.5.3 Validar si el error es aceptable para el negocio


El alcance declarado en 3.1.3 era comparar cuatro configuraciones y reportar RMSE, R2 y
MAE — eso ya se cumplio. Sobre el desempeño en si: compara el R2 del ganador contra el
umbral que exigiria un sistema en produccion real, y documenta tu conclusion aqui.


## Fase 6 — Deployment


## 3.6.1 Guardar el modelo seleccionado


In [27]:
modelo_ganador = modelo_rf  # ajusta esta linea segun tu propio resultado (3.5.2)

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ce_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_ce_regresion")


Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_ce_regresion


## Cierre


## 3.7.1 Documentar hallazgos y responder preguntas de reflexion


Agrega celdas markdown breves debajo de cada bloque de codigo relevante explicando que
hiciste y que observaste.

**Reflexion tecnica breve** (5 a 8 lineas): por que resolver los duplicados de variables
ambientales antes del `join` evita un problema mas dificil de rastrear despues? que
configuracion tuvo el mejor RMSE, y por cuanto margen supero a la linea base? cual fue la
variable con mayor `featureImportances`, y tiene sentido fisico? por que `VectorAssembler`
es un paso obligatorio en Spark MLlib y no en scikit-learn?
